# Run Graph-Mamba Inference

Inference parameters come from `configs/default.yaml`. The notebook reads the global `run_name`, loads `checkpoints/<run_name>/<checkpoint_filename>`, reads graph data from `output/<run_name>/`, and writes a compiled AirNow-like NetCDF file under `output/<run_name>/`.

In [1]:
from pathlib import Path
import copy
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'configs/default.yaml').exists():
    REPO_ROOT = Path('/mnt/data3/GraphMamba')

CONFIG_PATH = REPO_ROOT / 'configs/default.yaml'
sys.path.insert(0, str(REPO_ROOT / 'src'))

from graphmamba.config import load_config, resolve_inference_paths

def make_repo_path(value):
    path = Path(value)
    return str(path if path.is_absolute() else REPO_ROOT / path)

config = copy.deepcopy(load_config(CONFIG_PATH))
config['paths']['checkpoint_root'] = make_repo_path(config['paths']['checkpoint_root'])
config['paths']['output_root'] = make_repo_path(config['paths']['output_root'])
inference_cfg = config['inference']
inference_paths = resolve_inference_paths(config)
checkpoint_path = inference_paths['checkpoint_path']
graph_path = inference_paths['graph_path']
output_csv = inference_paths['output_csv']
output_netcdf = inference_paths['output_netcdf']

print(json.dumps({
    'config_path': str(CONFIG_PATH),
    'run_name': config['run_name'],
    'checkpoint_path': str(checkpoint_path),
    'graph_path': str(graph_path),
    'date_range': [inference_cfg['start_date'], inference_cfg['end_date']],
    'lead_time': inference_cfg.get('lead_time', config['data'].get('lead_time', 0)),
    'device': inference_cfg['device'],
    'multi_gpu': inference_cfg.get('multi_gpu', 'off'),
    'gpu_min_free_memory_gb': inference_cfg.get('gpu_min_free_memory_gb', 0.0),
    'gpu_max_count': inference_cfg.get('gpu_max_count'),
    'batch_size': inference_cfg['batch_size'],
    'show_progress': inference_cfg.get('show_progress', True),
    'output_csv': str(output_csv),
    'output_netcdf': str(output_netcdf),
}, indent=2))

{
  "config_path": "/mnt/data3/GraphMamba/configs/default.yaml",
  "run_name": "test",
  "checkpoint_path": "/mnt/data3/GraphMamba/checkpoints/test/last_model.pt",
  "graph_path": "/mnt/data3/GraphMamba/output/test/unified_graph.npz",
  "date_range": [
    "2024-07-01",
    "2024-09-30"
  ],
  "lead_time": 0,
  "device": "auto",
  "multi_gpu": true,
  "gpu_min_free_memory_gb": 20.0,
  "gpu_max_count": 4,
  "batch_size": 1,
  "show_progress": true,
  "output_csv": "/mnt/data3/GraphMamba/output/test/predictions.csv",
  "output_netcdf": "/mnt/data3/GraphMamba/output/test/predictions.nc"
}


In [2]:
from graphmamba.inference import load_inference_bundle

bundle = load_inference_bundle(
    checkpoint_path=checkpoint_path,
    graph_path=graph_path,
    device=inference_cfg['device'],
    multi_gpu=inference_cfg.get('multi_gpu', 'auto'),
    gpu_min_free_memory_gb=inference_cfg.get('gpu_min_free_memory_gb', 0.0),
    gpu_max_count=inference_cfg.get('gpu_max_count'),
)
train_config = bundle.config
lead_time = inference_cfg.get('lead_time', train_config['data'].get('lead_time', 0))
print(json.dumps({
    'device': str(bundle.device),
    'device_ids': bundle.device_ids,
    'target_mean': bundle.target_mean,
    'target_std': bundle.target_std,
    'lead_time': lead_time,
    'num_airnow': bundle.graph['metadata']['num_airnow'],
    'num_tempo': bundle.graph['metadata']['num_tempo'],
}, indent=2))

{
  "device": "cuda:6",
  "device_ids": [
    6,
    3,
    4,
    7
  ],
  "target_mean": 7.919435335776275,
  "target_std": 8.37903422674131,
  "lead_time": 0,
  "num_airnow": 191,
  "num_tempo": 933
}


In [3]:
from graphmamba.data import TempoAirNowDataset

dataset = TempoAirNowDataset(
    graph_path=graph_path,
    tempo_dir=train_config['paths']['tempo_dir'],
    airnow_dir=train_config['paths']['airnow_dir'],
    start_date=inference_cfg['start_date'],
    end_date=inference_cfg['end_date'],
    context_steps=train_config['data']['context_steps'],
    max_time_snap_hours=train_config['data']['max_time_snap_hours'],
    max_column=train_config['data']['max_column'],
    min_valid_targets=inference_cfg['min_valid_targets'],
    target_mean=bundle.target_mean,
    target_std=bundle.target_std,
    lead_time=lead_time,
)
print(json.dumps({
    'inference_windows': len(dataset),
    'lead_time': dataset.lead_time,
    'start_date': inference_cfg['start_date'],
    'end_date': inference_cfg['end_date'],
}, indent=2))

{
  "inference_windows": 1056,
  "lead_time": 0.0,
  "start_date": "2024-07-01",
  "end_date": "2024-09-30"
}


In [4]:
from graphmamba.inference import predict_dataframe

predictions = predict_dataframe(
    dataset,
    bundle,
    batch_size=inference_cfg['batch_size'],
    num_workers=inference_cfg['num_workers'],
    show_progress=inference_cfg.get('show_progress', True),
)
predictions.head()

Running inference: 100%|██████████| 1056/1056 [5:04:23<00:00, 17.29s/batch] 


,target_time,predictor_scan_time,lead_time,site_id,site_name,agency,latitude,longitude,predicted_no2_ppb,observed_no2_ppb,has_observation,absolute_error_ppb,delta_t_hours,nearest_tempo_distance_km
0,2024-07-01 17:00:00+00:00,2024-07-01 17:14:22+00:00,0.0,000081001,SWIFT CURRENT,Saskatchewan Environment,50.285831,-107.816887,3.458370,2.0,True,1.458370,1.0,0.674179
1,2024-07-01 17:00:00+00:00,2024-07-01 17:14:22+00:00,0.0,000100110,Kensington Park,Metro Vancouver,49.279400,-122.971100,17.815216,5.7,True,12.115216,1.0,1.048203
2,2024-07-01 17:00:00+00:00,2024-07-01 17:14:22+00:00,0.0,000100119,Burnaby South,Metro Vancouver,49.215279,-122.985558,11.522321,6.7,True,4.822321,1.0,0.669827
3,2024-07-01 17:00:00+00:00,2024-07-01 17:14:22+00:00,0.0,000100125,North Delta,Metro Vancouver,49.158329,-122.901672,8.237756,3.8,True,4.437756,1.0,1.106586
4,2024-07-01 17:00:00+00:00,2024-07-01 17:14:22+00:00,0.0,000100127,Surrey East,Metro Vancouver,49.132778,-122.694168,3.468951,3.6,True,0.131049,1.0,0.432686


In [5]:
from graphmamba.metrics import pearson_corr

observed = predictions[predictions['has_observation']].copy()
if len(observed):
    rmse = ((observed['predicted_no2_ppb'] - observed['observed_no2_ppb']) ** 2).mean() ** 0.5
    ss_res = ((observed['observed_no2_ppb'] - observed['predicted_no2_ppb']) ** 2).sum()
    ss_tot = ((observed['observed_no2_ppb'] - observed['observed_no2_ppb'].mean()) ** 2).sum()
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    dt_corr = pearson_corr(observed['delta_t_hours'].to_numpy(), observed['absolute_error_ppb'].to_numpy())
    distance_corr = pearson_corr(observed['nearest_tempo_distance_km'].to_numpy(), observed['absolute_error_ppb'].to_numpy())
    print(json.dumps({
        'observations': int(len(observed)),
        'rmse': float(rmse),
        'r2': float(r2),
        'abs_error_delta_t_corr': float(dt_corr),
        'abs_error_nearest_tempo_distance_corr': float(distance_corr),
    }, indent=2))
else:
    print('No observed AirNow targets are available in this inference range.')

{
  "observations": 187450,
  "rmse": 4.907973996313431,
  "r2": 0.5072251883845271,
  "abs_error_delta_t_corr": 0.04564890105523169,
  "abs_error_nearest_tempo_distance_corr": 0.010347143574882452
}


In [6]:
from graphmamba.inference import write_predictions_netcdf

netcdf_path = write_predictions_netcdf(
    predictions,
    output_netcdf,
    run_name=config['run_name'],
    attrs={
        'checkpoint_path': checkpoint_path,
        'graph_path': graph_path,
        'lead_time': lead_time,
        'inference_start_date': inference_cfg['start_date'],
        'inference_end_date': inference_cfg['end_date'],
    },
)
output_csv.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(output_csv, index=False)
print({'netcdf': str(netcdf_path), 'csv': str(output_csv)})

{'netcdf': '/mnt/data3/GraphMamba/output/test/predictions.nc', 'csv': '/mnt/data3/GraphMamba/output/test/predictions.csv'}
